In [1]:
import pandas as pd
import pandas as pd
# Open the TSV file for reading
with open('ASHA_New_Basic_Report.tsv', 'r') as file:
    # Initialize variables to store extracted information
    cancer_type = None
    sample_id = None
    biomarkers_data = []
    hrr_data = []
    read_biomarkers = False  # Flag to indicate when to start reading biomarkers data
    read_hrr = False  # Flag to indicate when to start reading HRR details
    spacing_encountered_biomarkers = False

    # Initialize variables for DNA Sequence Variants and Copy Number Variations
    dna_sequence_variants_data = []
    copy_number_variations_data = []
    reading_dna_sequence_variants = False
    reading_copy_number_variations = False
    signal_line_space_count = 0

    # Iterate through each line in the file
    for line in file:
        # Strip leading and trailing whitespaces
        line = line.strip()

        # Extract Sample Cancer Type, Sample ID, and Tumor Mutational Burden
        if line.startswith("Sample Cancer Type"):
            cancer_type = line.split('\t')[1]
            print("Sample Cancer Type:", cancer_type)
        elif line.lower().startswith("**sample id:**|"):
            sample_id = line.split('|')[1].strip()
            print("Sample ID:", sample_id)
        elif line.startswith("Tumor Mutational Burden"):
            mutational_burden = line.split('\t')[1]
            print("Tumor Mutational Burden:", mutational_burden)

        # Start reading biomarkers data when encountering the header
        elif line == "Relevant Biomarkers":
            read_biomarkers = True
            continue
        
        # Skip one line space between header and data for Biomarkers
        elif read_biomarkers and not spacing_encountered_biomarkers:
            spacing_encountered_biomarkers = True
            continue

        # Extract data under Relevant Biomarkers and form a table
        elif read_biomarkers and line:
            biomarkers_data.append(line.split('\t'))

        # Stop reading biomarkers data when an empty line is reached
        elif read_biomarkers and spacing_encountered_biomarkers and not line:
            read_biomarkers = False

        # Start reading HRR details when encountering the header
        elif line == "HRR Details":
            read_hrr = True
            continue

        # Skip lines containing additional information under HRR Details
        elif read_hrr and ("Homologous recombination repair (HRR) genes were defined" in line or "Disclaimer" in line):
            read_hrr = False
            continue

        # Extract data under HRR Details and form a table
        elif read_hrr and line:
            hrr_data.append(line.split('\t'))

        # Check if the line contains the header for DNA Sequence Variants
        elif line == "DNA Sequence Variants":
            reading_dna_sequence_variants = True
            reading_copy_number_variations = False
            signal_line_space_count = 0
            continue

        # Check if the line contains the header for Copy Number Variations
        elif line == "Copy Number Variations":
            reading_copy_number_variations = True
            reading_dna_sequence_variants = False
            signal_line_space_count = 0
            continue

        # Check for the signal line space and stop reading Copy Number Variations
        elif not line:
            signal_line_space_count += 1
            if signal_line_space_count == 2:
                reading_copy_number_variations = False
                continue

        # Extract data under DNA Sequence Variants section
        elif reading_dna_sequence_variants and line:
            dna_sequence_variants_data.append(line.split('\t'))

        # Extract data under Copy Number Variations section
        elif reading_copy_number_variations and line:
            copy_number_variations_data.append(line.split('\t'))

# Convert the extracted data into DataFrames
biomarkers_columns = ["Genomic Alteration","Tier", "Allele Frequency", "Mut/WT Ratio", 
           "Locus", "Transcript","Relevant Therapies In this cancer type", 
           "Relevant Therapies In other cancer type", "Clinical Trials", 
           "Prognostic significance", "Diagnostic significance"]
biomarkers_df = pd.DataFrame(columns=biomarkers_columns, data=biomarkers_data[1:])

hrr_columns = ["Gene/Genomic Alteration", "Finding"]
hrr_df = pd.DataFrame(columns=hrr_columns, data=hrr_data[1:])

dna_sequence_variants_df = pd.DataFrame(dna_sequence_variants_data[1:], columns=dna_sequence_variants_data[0])
dna_sequence_variants_df=dna_sequence_variants_df.drop(columns=['Tier', 'Transcript', 'Allele Frequency', 'Variant ID'])
dna_sequence_variants_merged_df = pd.merge(biomarkers_df, dna_sequence_variants_df, on="Locus")
copy_number_variations_df = pd.DataFrame(copy_number_variations_data[1:], columns=copy_number_variations_data[0])
copy_number_variations_df=copy_number_variations_df.drop(columns=['Tier'])
copy_number_variations_merged_df = pd.merge(biomarkers_df, copy_number_variations_df, on="Locus")
# Read the lines and filter out those starting with '##' for filtered_in_df
with open('ASHA_Filtered_In.tsv', 'r') as file:
    lines = [line.strip() for line in file if not line.startswith('##')]

# Construct DataFrame from the filtered lines for filtered_in_df
filtered_in_df = pd.DataFrame([line.split('\t') for line in lines])

# Set the first row as column headers
filtered_in_df.columns = filtered_in_df.iloc[0]

# Remove the first row since it's now the header
filtered_in_df = filtered_in_df[1:]

# Select columns of interest
selected_columns = ['Locus', 'Oncomine Gene Class', 'Variant ID', 'ExAC GAF', 'dbSNP']
filtered_in_df = filtered_in_df[selected_columns]

# Read the lines and filter out those starting with '##' for filtered_out_df
with open('ASHA_Filtered_Out.tsv', 'r') as file:
    lines = [line.strip() for line in file if not line.startswith('##')]

# Construct DataFrame from the filtered lines for filtered_out_df
filtered_out_df = pd.DataFrame([line.split('\t') for line in lines])

# Set the first row as column headers
filtered_out_df.columns = filtered_out_df.iloc[0]

# Remove the first row since it's now the header
filtered_out_df = filtered_out_df[1:]

# Select columns of interest
filtered_out_df = filtered_out_df[selected_columns]

# Merge DNA Sequence Variants with filtered_in_df and filtered_out_df
dna_sequence_variants_merged_with_filter_in_df = pd.merge(dna_sequence_variants_merged_df, filtered_in_df, on='Locus')
dna_sequence_variants_merged_with_filter_out_df = pd.merge(dna_sequence_variants_merged_df, filtered_out_df, on='Locus')
dna_sequence_variants_merged_with_filter_df = pd.concat([dna_sequence_variants_merged_with_filter_in_df, dna_sequence_variants_merged_with_filter_out_df])
dna_sequence_variants_merged_with_filter_df = dna_sequence_variants_merged_with_filter_df.drop_duplicates(subset=['Locus'])
# Merge Copy Number Variations with filtered_in_df and filtered_out_df
copy_number_variations_merged_with_filter_in_df = pd.merge(copy_number_variations_merged_df, filtered_in_df, on='Locus')
copy_number_variations_merged_with_filter_out_df = pd.merge(copy_number_variations_merged_df, filtered_out_df, on='Locus')
copy_number_variations_merged_with_filter_df = pd.concat([copy_number_variations_merged_with_filter_in_df, copy_number_variations_merged_with_filter_out_df])
copy_number_variations_merged_with_filter_df = copy_number_variations_merged_with_filter_df.drop_duplicates(subset=['Locus'])
# Print the results
print("DNA Sequence Variants - Filtered:")
print(dna_sequence_variants_merged_with_filter_df)
dna_sequence_variants_merged_with_filter_df.to_excel('dna_sequence_variants.xlsx', index=False)
print("-------------------------------------------")
print("Copy Number Variations - Filtered:")
print(copy_number_variations_merged_with_filter_df)
copy_number_variations_merged_with_filter_df.to_excel('copy_number_variants.xlsx', index=False)


Sample Cancer Type: Endometrial Carcinoma
Sample ID: AK1002392381
Tumor Mutational Burden: 1.92 Mut/Mb measured
DNA Sequence Variants - Filtered:
  Genomic Alteration Tier Allele Frequency Mut/WT Ratio           Locus  \
0       PIK3CA E545K  IIC           69.02%               chr3:178936091   
1         TP53 G244D   IA           67.32%                chr17:7577550   
2        FBXW7 R465H  IIC           21.55%               chr4:153249384   
1  BRIP1 c.1474-3T>C  IIC           38.21%               chr17:59861788   

    Transcript Relevant Therapies In this cancer type  \
0  NM_006218.4                                   None   
1  NM_000546.6                                   None   
2  NM_033632.3                                   None   
1  NM_032043.3                                   None   

             Relevant Therapies In other cancer type Clinical Trials  \
0  alpelisib + hormone therapy [1,2], capivaserti...              23   
1                                               